# 🎓 Personalized AI Learning Tutor

A GenAI-inspired tutor that adapts learning paths to learner goals, skill level, quiz performance, and available study time. This reproducible baseline uses Python's standard library and does not require a paid API key.

> AI recommendations are suggestions. Learners should be able to review and customize their study plans.


## ✨ Features
- 👤 Learner profile and goals
- 🧭 Level-aware learning path
- 📚 Lesson cards and examples
- 📝 Quiz evaluation
- 📊 Progress tracking and skill-gap signals
- 🔁 Adaptive practice recommendations
- 🗓️ Personalized weekly study plan
- 🧠 Optional LLM/RAG extension
- 📤 JSON progress report export

## 🏗️ Architecture
```text
Learner Profile → Learning Path → Lesson → Quiz → Progress Analysis
       ↑                                           ↓
       └──────── Adaptive Recommendations ← Study Plan
```


In [ ]:
from dataclasses import asdict
from collections import defaultdict
from pathlib import Path
import json

print("Tutor notebook ready ✅")


## 👤 1. Create a learner profile

In [ ]:
learner = {
    "name": "Alex",
    "goal": "Learn Python and Generative AI",
    "current_level": "beginner",  # beginner, intermediate, advanced
    "weekly_study_hours": 5,
    "preferred_style": "examples and hands-on practice",
    "target_weeks": 4,
}
learner


## 📚 2. Define the curriculum

In [ ]:
curriculum = [
    {"topic": "Python Fundamentals", "level": "beginner", "hours": 3,
     "prerequisites": [], "objectives": ["Understand variables", "Use conditions and loops"],
     "lesson": "Variables store values. Conditions choose paths, while loops repeat work.",
     "example": "for number in range(3):\n    print(number)"},
    {"topic": "Functions and Modules", "level": "beginner", "hours": 2,
     "prerequisites": ["Python Fundamentals"], "objectives": ["Define reusable functions", "Import modules"],
     "lesson": "Functions package reusable behavior; modules organize code.",
     "example": "def greet(name):\n    return f'Hello, {name}!'"},
    {"topic": "Pandas Data Handling", "level": "intermediate", "hours": 3,
     "prerequisites": ["Python Fundamentals"], "objectives": ["Load tabular data", "Filter and summarize rows"],
     "lesson": "Pandas DataFrames help load, filter, transform, and summarize tabular data.",
     "example": "import pandas as pd\ndf = pd.DataFrame({'score': [70, 90]})\nprint(df['score'].mean())"},
    {"topic": "LLM and Prompt Engineering", "level": "intermediate", "hours": 3,
     "prerequisites": ["Python Fundamentals"], "objectives": ["Explain tokens and context", "Write clear prompts"],
     "lesson": "An LLM generates text from context. Clear instructions and examples can improve responses.",
     "example": "prompt = 'Explain recursion to a beginner using one example.'"},
    {"topic": "Retrieval-Augmented Generation", "level": "advanced", "hours": 5,
     "prerequisites": ["LLM and Prompt Engineering"], "objectives": ["Explain retrieval plus generation", "Ground answers in sources"],
     "lesson": "RAG retrieves relevant documents and supplies them as context to a language model.",
     "example": "question → retrieve passages → build context → generate grounded answer"},
]
len(curriculum)


## 🧭 3. Recommend a learning path

In [ ]:
LEVEL_ORDER = {"beginner": 0, "intermediate": 1, "advanced": 2}

def recommend_learning_path(profile, topics):
    max_level = LEVEL_ORDER.get(profile["current_level"], 0)
    return [t for t in topics if LEVEL_ORDER[t["level"]] <= max_level + 1]

learning_path = recommend_learning_path(learner, curriculum)
for i, topic in enumerate(learning_path, 1):
    print(f"{i}. {topic['topic']} ({topic['level']}, about {topic['hours']} hours)")


## 📖 4. Generate a lesson card

In [ ]:
def lesson_card(topic, profile):
    return {
        "title": topic["topic"],
        "learner_level": profile["current_level"],
        "objectives": topic["objectives"],
        "explanation": topic["lesson"],
        "worked_example": topic["example"],
        "practice_prompt": f"Create an exercise for: {topic['objectives'][0].lower()}",
        "estimated_hours": topic["hours"],
    }

print(json.dumps(lesson_card(learning_path[0], learner), indent=2))


## 📝 5. Quiz bank

In [ ]:
quiz_bank = {
    "Python Fundamentals": [
        {"question": "Which symbol starts a Python comment?", "options": ["//", "#", "<!--", "/*"], "answer": "#",
         "explanation": "A single-line Python comment starts with #."},
        {"question": "What does range(3) yield when iterated?", "options": ["1,2,3", "0,1,2", "0,1,2,3", "3 only"], "answer": "0,1,2",
         "explanation": "range(3) yields 0, 1, and 2."},
    ],
    "Functions and Modules": [
        {"question": "Which keyword defines a function?", "options": ["func", "define", "def", "function"], "answer": "def",
         "explanation": "Python uses def to define functions."},
    ],
    "LLM and Prompt Engineering": [
        {"question": "What is a prompt?", "options": ["Instructions/context for a model", "A database", "A compiler", "A GPU"],
         "answer": "Instructions/context for a model", "explanation": "A prompt supplies instructions and context to a model."},
    ],
    "Retrieval-Augmented Generation": [
        {"question": "What is retrieval used for in RAG?", "options": ["Find relevant source context", "Replace testing", "Encrypt a model", "Compile code"],
         "answer": "Find relevant source context", "explanation": "Retrieval supplies relevant information to ground generation."},
    ],
}
quiz_bank


## ✅ 6. Evaluate quiz answers and record progress

In [ ]:
progress = defaultdict(list)

def evaluate_quiz(topic_name, answers):
    questions = quiz_bank.get(topic_name, [])
    results, correct = [], 0
    for index, q in enumerate(questions):
        selected = answers.get(index)
        ok = selected == q["answer"]
        correct += int(ok)
        results.append({
            "question": q["question"], "selected": selected,
            "correct_answer": q["answer"], "is_correct": ok,
            "explanation": q["explanation"],
        })
    score = round(100 * correct / len(questions), 1) if questions else 0.0
    progress[topic_name].append(score)
    return {"topic": topic_name, "score_percent": score, "results": results}

demo_result = evaluate_quiz("Python Fundamentals", {0: "#", 1: "0,1,2"})
demo_result


## 📊 7. Adaptive learning recommendations

In [ ]:
def learning_recommendation(score):
    if score < 60:
        return {"status": "needs_revision", "recommendation": "Review the lesson and example, then retry a short quiz."}
    if score < 80:
        return {"status": "practice_more", "recommendation": "Try two more exercises and explain the concept in your own words."}
    return {"status": "ready_to_progress", "recommendation": "Move to the next topic and revisit this topic during spaced review."}

learning_recommendation(demo_result["score_percent"])


## 🗓️ 8. Build a weekly study plan

In [ ]:
def build_study_plan(profile, topics):
    remaining = max(1, int(profile["weekly_study_hours"]))
    schedule = []
    for topic in topics:
        if remaining <= 0:
            break
        allocated = min(remaining, max(1, min(2, topic["hours"])))
        schedule.append({
            "topic": topic["topic"], "hours_this_week": allocated,
            "activity": "Learn concept → study example → practice → self-check",
        })
        remaining -= allocated
    return {"learner": profile["name"], "weekly_hours": profile["weekly_study_hours"], "schedule": schedule}

study_plan = build_study_plan(learner, learning_path)
print(json.dumps(study_plan, indent=2))


## 🧠 9. Optional GenAI tutor extension

Use an LLM to produce alternative explanations, hints, examples, and quiz drafts. Keep API keys in environment variables and do not commit them.

Suggested system prompt:
```text
You are a supportive tutor. Explain concepts at the learner's level using the
supplied course context. Give one example and one practice question. If context
is insufficient, say so rather than inventing facts. Do not reveal quiz answers
until the learner attempts the question. Return structured JSON.
```

For a stronger version, use RAG over approved course materials, validate generated quiz answer keys, and keep source references with factual explanations.


## 📤 10. Export progress report

In [ ]:
learner_report = {
    "learner_profile": learner,
    "recommended_topics": [t["topic"] for t in learning_path],
    "quiz_history": dict(progress),
    "study_plan": study_plan,
    "next_step": learning_recommendation(demo_result["score_percent"]),
}
report_path = Path("learning_tutor_progress.json")
report_path.write_text(json.dumps(learner_report, indent=2), encoding="utf-8")
print("Progress report saved to:", report_path.resolve())


## 📏 Evaluation Metrics
- 🎯 Quiz correctness and learning gain
- 🔁 Topic mastery across repeated attempts
- 🧭 Recommendation relevance
- 👩‍🎓 Learner satisfaction and completion rate
- ⚖️ Fairness across learner groups
- ⏱️ Response latency and 💰 model cost

## 🛡️ Responsible AI and privacy
- Collect only learner data needed for tutoring.
- Make progress tracking transparent and allow correction/deletion.
- Do not infer sensitive traits from performance.
- Validate AI-generated explanations and quiz answer keys.
- Offer accessible content and multiple ways to practice.
- Treat learning-style preferences as adjustable preferences, not fixed traits.
- Protect API keys and minimize personal data in logs.

## 🗺️ Roadmap
- [x] Learner profile and curriculum
- [x] Learning path recommendations
- [x] Lesson cards and quiz evaluation
- [x] Adaptive practice suggestions
- [x] Weekly study plan and JSON export
- [ ] LLM-backed explanations and quiz drafts
- [ ] RAG over course documents
- [ ] Spaced repetition and mastery tracking
- [ ] Streamlit / React interface
- [ ] Authentication and persistent database
- [ ] Evaluation dashboard

## 💼 Resume-ready description
Built a personalized AI learning tutor prototype that recommends level-aware learning paths, presents lesson cards, evaluates quizzes, identifies skill gaps, generates weekly study plans, and exports learner progress, with an extensible architecture for LLM tutoring and course-material RAG.
